In [57]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split,GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import classification_report,accuracy_score,f1_score,recall_score,roc_auc_score



In [58]:
df = pd.read_csv("../dataset/Telco-Customer-Churn.csv")
df. head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [59]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [60]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. Remove customer ID
df = df.drop(columns=["customerID"], errors="ignore").copy()

# 2. Clean TotalCharges
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype(str).str.strip(),
    errors="coerce"
)

# 3. Clean target column
df["Churn"] = (
    df["Churn"].astype(str).str.strip()
    .map({"Yes": 1, "No": 0})
)

# Remove rows where target is missing or invalid
df = df.dropna(subset=["Churn"]).copy()
df["Churn"] = df["Churn"].astype(int)

# 4. Selected original features
selected_features = [
    "tenure",
    "TotalCharges",
    "MonthlyCharges",
    "InternetService",
    "Contract",
    "PaymentMethod",
    "OnlineSecurity",
    "TechSupport",
    "OnlineBackup",
    "PaperlessBilling",
    "StreamingTV",
    "DeviceProtection",
    "StreamingMovies"
]

X = df[selected_features].copy()
y = df["Churn"].copy()

# 5. Separate numeric and categorical features
numeric_features = [
    "tenure",
    "TotalCharges",
    "MonthlyCharges"
]

categorical_features = [
    col for col in selected_features
    if col not in numeric_features
]

# 6. Split BEFORE fitting preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 7. Numeric preprocessing: missing values + scaling
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 8. Categorical preprocessing: missing values + encoding
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        drop="first",
        handle_unknown="ignore"
    ))
])

# 9. Combine both preprocessing pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed!")
print("Original selected features:", len(selected_features))
print("Encoded training shape:", X_train_processed.shape)
print("Encoded testing shape:", X_test_processed.shape)
print("Missing target values:", y.isna().sum())

Preprocessing completed!
Original selected features: 13
Encoded training shape: (5634, 23)
Encoded testing shape: (1409, 23)
Missing target values: 0


In [61]:

joblib.dump(
    preprocessor,
    "../models/churn_preprocessor.pkl"
)

joblib.dump(
    selected_features,
    "../models/churn_selected_features.pkl"
)

print("Preprocessor and selected features saved successfully!")

Preprocessor and selected features saved successfully!


In [62]:

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=42,
        class_weight="balanced"
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42, class_weight="balanced"
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

trained_models = {}
results = []

for name, classifier in models.items():

    # Same preprocessing for every model
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", classifier)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(
            precision_score(y_test, y_pred, zero_division=0), 4
        ),
        "Recall": round(
            recall_score(y_test, y_pred, zero_division=0), 4
        ),
        "F1 Score": round(
            f1_score(y_test, y_pred, zero_division=0), 4
        ),
        "ROC-AUC": round(roc_auc_score(y_test, y_prob), 4)
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="ROC-AUC", ascending=False
)

print(results_df.to_string(index=False))

              Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
  Gradient Boosting    0.7949     0.6412  0.5160    0.5719   0.8407
Logistic Regression    0.7921     0.6294  0.5267    0.5735   0.8403
      Random Forest    0.7771     0.6034  0.4679    0.5271   0.8135
      Decision Tree    0.7374     0.5053  0.5053    0.5053   0.6628


In [63]:

print(
    results_df[
        ["Model", "Accuracy", "Recall", "F1 Score", "ROC-AUC"]
    ].to_string(index=False)
)

print("\nChurn distribution:")
print(y.value_counts(normalize=True).rename("Percentage"))

              Model  Accuracy  Recall  F1 Score  ROC-AUC
  Gradient Boosting    0.7949  0.5160    0.5719   0.8407
Logistic Regression    0.7921  0.5267    0.5735   0.8403
      Random Forest    0.7771  0.4679    0.5271   0.8135
      Decision Tree    0.7374  0.5053    0.5053   0.6628

Churn distribution:
Churn
0    0.73463
1    0.26537
Name: Percentage, dtype: float64


In [64]:
import os
import joblib

# Project ke andar models folder create hoga
os.makedirs("../models", exist_ok=True)

# Best model pipeline select karo
best_model = trained_models["Gradient Boosting"]

# Complete pipeline save karo
joblib.dump(best_model, "../models/churn_pipeline.pkl")

print("Model saved successfully!")
print("Model: Gradient Boosting")
print("File: models/churn_pipeline.pkl")

Model saved successfully!
Model: Gradient Boosting
File: models/churn_pipeline.pkl


In [65]:
import joblib
import pandas as pd

# Saved pipeline load karo
loaded_model = joblib.load("../models/churn_pipeline.pkl")

# Test data se ek customer lo
sample_customer = X_test.iloc[[0]]

# Prediction
prediction = loaded_model.predict(sample_customer)[0]

# Churn probability
probability = loaded_model.predict_proba(sample_customer)[0][1]

print("Customer Details:")
print(sample_customer.to_string(index=False))

print("\nPrediction:", "Will Churn" if prediction == 1 else "Will Not Churn")
print(f"Churn Probability: {probability * 100:.2f}%")

Customer Details:
 tenure  TotalCharges  MonthlyCharges InternetService Contract           PaymentMethod OnlineSecurity TechSupport OnlineBackup PaperlessBilling StreamingTV DeviceProtection StreamingMovies
     72        8468.2          114.05     Fiber optic Two year Credit card (automatic)            Yes         Yes          Yes              Yes         Yes              Yes             Yes

Prediction: Will Not Churn
Churn Probability: 2.17%


In [66]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    roc_auc_score
)

import pandas as pd

# 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Models and hyperparameters
model_grids = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, random_state=42),
        {
            "model__C": [0.01, 0.1, 1.0, 10.0],
            "model__solver": ["liblinear", "lbfgs"],
            "model__class_weight": [None, "balanced"]
        }
    ),

    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {
            "model__max_depth": [3, 5, 10, None],
            "model__min_samples_split": [2, 5, 10],
            "model__class_weight": [None, "balanced"]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [5, 10, 15],
            "model__min_samples_split": [2, 5],
            "model__class_weight": [None, "balanced"]
        }
    ),

    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=42),
        {
            "model__n_estimators": [100, 150],
            "model__learning_rate": [0.01, 0.1],
            "model__max_depth": [3, 5]
        }
    )
}

results = []
best_estimators = {}

for name, (classifier, grid) in model_grids.items():

    # Preprocessing + model in one pipeline
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", classifier)
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=grid,
        cv=cv,
        scoring="f1",
        n_jobs=-1,
        refit=True
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_estimators[name] = best_model

    # Evaluate on the same held-out test set
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Best Parameters": grid_search.best_params_,
        "CV F1-Score": round(grid_search.best_score_, 4),
        "Test F1-Score": round(
            f1_score(y_test, y_pred, zero_division=0), 4
        ),
        "Test Recall": round(
            recall_score(y_test, y_pred, zero_division=0), 4
        ),
        "Test ROC-AUC": round(
            roc_auc_score(y_test, y_proba), 4
        ),
        "Test Accuracy": round(
            accuracy_score(y_test, y_pred), 4
        )
    })

comparison_df = (
    pd.DataFrame(results)
    .sort_values("CV F1-Score", ascending=False)
    .reset_index(drop=True)
)

print(comparison_df[
    [
        "Model",
        "CV F1-Score",
        "Test F1-Score",
        "Test Recall",
        "Test ROC-AUC",
        "Test Accuracy"
    ]
].to_string(index=False))

print("\nBest parameters for each model:")
for name, row in comparison_df.iterrows():
    print(f"\n{row['Model']}: {row['Best Parameters']}")

              Model  CV F1-Score  Test F1-Score  Test Recall  Test ROC-AUC  Test Accuracy
      Random Forest       0.6358         0.6230       0.7380        0.8381         0.7630
Logistic Regression       0.6249         0.6168       0.7941        0.8397         0.7381
      Decision Tree       0.6005         0.6036       0.8021        0.8268         0.7204
  Gradient Boosting       0.5877         0.5719       0.5160        0.8407         0.7949

Best parameters for each model:

Random Forest: {'model__class_weight': 'balanced', 'model__max_depth': 10, 'model__min_samples_split': 5, 'model__n_estimators': 100}

Logistic Regression: {'model__C': 10.0, 'model__class_weight': 'balanced', 'model__solver': 'lbfgs'}

Decision Tree: {'model__class_weight': 'balanced', 'model__max_depth': 5, 'model__min_samples_split': 2}

Gradient Boosting: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}


In [76]:
import os
import joblib

# Highest Test Accuracy wala model select karo
best_row = comparison_df.loc[comparison_df["Test Accuracy"].idxmax()]

best_model_name = best_row["Model"]
best_model = best_estimators[best_model_name]

print("Best Model:", best_model_name)
print("Best Parameters:", best_row["Best Parameters"])
print("Test Accuracy:", best_row["Test Accuracy"])
print("Test F1-Score:", best_row["Test F1-Score"])
print("Test Recall:", best_row["Test Recall"])
print("Test ROC-AUC:", best_row["Test ROC-AUC"])

# Model save karo
os.makedirs("../models", exist_ok=True)

joblib.dump(best_model, "../models/churn_pipeline.pkl")

print("\nBest model saved successfully!")
print("Path: ../models/churn_pipeline.pkl")

Best Model: Gradient Boosting
Best Parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}
Test Accuracy: 0.7949
Test F1-Score: 0.5719
Test Recall: 0.516
Test ROC-AUC: 0.8407

Best model saved successfully!
Path: ../models/churn_pipeline.pkl


In [77]:
import joblib

loaded_model = joblib.load("../models/churn_pipeline.pkl")

y_pred_saved = loaded_model.predict(X_test)

print(
    "Saved Model Accuracy:",
    round(accuracy_score(y_test, y_pred_saved), 4)
)

Saved Model Accuracy: 0.7949


In [82]:
import joblib
import pandas as pd

# Load complete saved pipeline
model = joblib.load("../models/churn_pipeline.pkl")

# Customer input: original, unprocessed values
new_customer = pd.DataFrame([{
    "tenure": 5,
    "TotalCharges": 250.50,
    "MonthlyCharges": 50.10,
    "InternetService": "DSL",
    "Contract": "Month-to-month",
    "PaymentMethod": "Electronic check",
    "OnlineSecurity": "No",
    "TechSupport": "No",
    "OnlineBackup": "Yes",
    "PaperlessBilling": "Yes",
    "StreamingTV": "No",
    "DeviceProtection": "No",
    "StreamingMovies": "No"
}])

# Preprocessing happens automatically inside the pipeline
prediction = model.predict(new_customer)[0]
probability = model.predict_proba(new_customer)[0][1]

print("Prediction:",
      "Churn" if prediction == 1 else "No Churn")
print(f"Churn Probability: {probability * 100:.2f}%")

Prediction: Churn
Churn Probability: 50.50%


In [23]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [47]:
df['Churn'].unique()

array([0, 1])